# Stage 4 — Hybrid Agent (BERT + LLM + Tavily)

**Цель:** прогнать low-confidence примеры val через LangGraph-агента и оценить гибридную систему (BERT на уверенных + LLM на пограничных).

**Вход (пути из `utils/config.py`):**
- `VAL_MERGED_PREDS_PATH` — val после Stage 3
- `STAGE3_COMPARISON_PATH` — порог `confidence_threshold` из Stage 3

**Выход:**
- `AGENT_LOW_CONF_PREDS_PATH`
- `HYBRID_VAL_PREDS_PATH`
- `STAGE4_REPORT_PATH`

**Перед запуском** — в `.env` в корне проекта:
```
VSEGPT_API_KEY=...
TAVILY_API_KEY=...
```

> Для первого прогона установи `SAMPLE_N = 5` (см. ячейку ниже). Для полного прогона — `SAMPLE_N = None`.

## 0. Зависимости

In [1]:
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import (
    AGENT_LLM_MODEL,
    AGENT_LOW_CONF_PREDS_PATH,
    BASE_DIR,
    COL_BERT_PROBA1,
    COL_FINAL_PRED,
    COL_ROUTED_TO,
    COL_SEARCH_USED,
    ENV_PATH,
    HYBRID_VAL_PREDS_PATH,
    STAGE3_COMPARISON_PATH,
    STAGE4_REPORT_PATH,
    VAL_MERGED_PREDS_PATH,
    create_directories,
)
from utils.bert_routing import load_confidence_threshold, max_confidence_series
from utils.stage4_agent import run_stage4

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
create_directories()
load_dotenv(ENV_PATH)

print('OK — project root:', BASE_DIR)
print('VSEGPT key:', 'set' if os.getenv('VSEGPT_API_KEY') or os.getenv('OPENAI_API_KEY') else 'MISSING')
print('Tavily key:', 'set' if os.getenv('TAVILY_API_KEY') else 'MISSING')

C:\Users\anast\miniconda3\envs\yandex-relevance-project\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


OK — project root: D:\My_courses\NLP_ODS_2026\yandex_relevance
VSEGPT key: set
Tavily key: set


## 1. Параметры запуска

| Параметр | Тест | Полный прогон |
|----------|------|---------------|
| `SAMPLE_N` | `5` | `None` |
| `SLEEP_SEC` | `0.3` | `0.3` |

При `SAMPLE_N = 5` берётся случайная подвыборка из low-confidence (seed=42). Метрики гибрида на val будут приблизительными — только для smoke-test.

In [2]:
# --- измени здесь ---
SAMPLE_N = None                # None = все low-confidence (число в §2 после Stage 3)
SLEEP_SEC = 0.3               # пауза между запросами к VseGPT
LLM_MODEL = AGENT_LLM_MODEL # deepseek/deepseek-v4-flash-alt по умолчанию
OVERWRITE = True              # False — не перезаписывать существующий parquet

mode = f'test ({SAMPLE_N} samples)' if SAMPLE_N else 'full'
print(f'Режим: {mode}')
print(f'LLM model: {LLM_MODEL}')

Режим: full
LLM model: deepseek/deepseek-v4-flash-alt


## 2. Проверка входных артефактов

In [3]:
if not Path(STAGE3_COMPARISON_PATH).exists():
    raise FileNotFoundError(f'Run Stage 3 first: {STAGE3_COMPARISON_PATH}')
if not Path(VAL_MERGED_PREDS_PATH).exists():
    raise FileNotFoundError(f'Run Stage 3 first: {VAL_MERGED_PREDS_PATH}')

with open(STAGE3_COMPARISON_PATH, encoding='utf-8') as f:
    stage3 = json.load(f)

THRESHOLD = load_confidence_threshold()
val_merged = pd.read_parquet(VAL_MERGED_PREDS_PATH)

bert_max = max_confidence_series(val_merged[COL_BERT_PROBA1])
n_low = int((bert_max < THRESHOLD).sum())
n_low_stage3 = stage3.get('low_confidence', {}).get('low_confidence_n')

print(f'Threshold (Stage 3): {THRESHOLD:.2f}')
print(f'Val rows:            {len(val_merged)}')
print(f'Low-confidence:      {n_low} ({100 * n_low / len(val_merged):.1f}%)')
if n_low_stage3 is not None and n_low != n_low_stage3:
    print(f'  WARNING: comparison.json low_confidence_n={n_low_stage3} != recount {n_low}')
print(f'Will run on:         {SAMPLE_N if SAMPLE_N else n_low} examples')

Threshold (Stage 3): 0.68
Val rows:            4558
Low-confidence:      925 (20.3%)
Will run on:         925 examples


## 3. Запуск агента

Каждый пример проходит через граф: `bert_route` → `decide_search` → (опционально) Tavily → `classify`.

На low-confidence подмножестве узел `bert_route` всегда направляет в LLM (порог уже отфильтрован выше).

In [4]:
summary = run_stage4(
    sample_n=SAMPLE_N,
    sleep_sec=SLEEP_SEC,
    llm_model=LLM_MODEL,
    overwrite=OVERWRITE,
)

print('\n=== Stage 4 complete ===')
print(f'  Threshold:            {summary["threshold"]:.2f}')
print(f'  Processed (low-conf):   {summary["low_conf_n"]}')
print(f'  Agent acc (low-conf):   {summary["agent_acc_on_low_conf"]:.4f}')
print(f'  Hybrid acc (val):       {summary["hybrid_acc"]:.4f}')
print(f'  Hybrid macro-F1:        {summary["hybrid_macro_f1"]:.4f}')
print(f'  Agent preds:            {summary["paths"].agent_preds_path}')
print(f'  Report:                 {summary["paths"].report_path}')

2026-06-02 20:10:34,350 - utils.stage4_agent - INFO - Stage 4: threshold=0.68, low-confidence=925 / 4558 (20.3%)
  0%|                                                                                          | 0/925 [00:00<?, ?it/s]2026-06-02 20:10:39,907 - httpx - INFO - HTTP Request: POST https://api.vsegpt.ru/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 20:10:43,072 - httpx - INFO - HTTP Request: POST https://api.vsegpt.ru/v1/chat/completions "HTTP/1.1 200 OK"
  0%|                                                                                | 1/925 [00:09<2:19:01,  9.03s/it]2026-06-02 20:10:47,402 - httpx - INFO - HTTP Request: POST https://api.vsegpt.ru/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 20:10:51,314 - httpx - INFO - HTTP Request: POST https://api.vsegpt.ru/v1/chat/completions "HTTP/1.1 200 OK"
  0%|▏                                                                               | 2/925 [00:17<2:11:42,  8.56s/it]2026-06-02 20:10:53,410 - httpx - INFO - HTTP Requ


=== Stage 4 complete ===
  Threshold:            0.68
  Processed (low-conf):   925
  Agent acc (low-conf):   0.5849
  Hybrid acc (val):       0.7738
  Hybrid macro-F1:        0.7725
  Agent preds:            D:\My_courses\NLP_ODS_2026\yandex_relevance\predictions\agent_low_conf_preds.parquet
  Report:                 D:\My_courses\NLP_ODS_2026\yandex_relevance\reports\stage4_agent\agent_metrics.json


## 4. Отчёт и предсказания

In [5]:
with open(STAGE4_REPORT_PATH, encoding="utf-8") as f:
    report = json.load(f)

def _fmt(v, ndigits=4):
  if v is None:
    return "—"
  if isinstance(v, float):
    return f"{v:.{ndigits}f}"
  return str(v)

def print_stage4_report(r: dict) -> None:
  delta_acc = r["hybrid_acc"] - r["bert_only_acc"]
  delta_f1 = r["hybrid_macro_f1"] - r["bert_only_macro_f1"]

  print("=" * 60)
  print("Stage 4 — общий отчёт (val)")
  print("=" * 60)
  print(f"Порог RuModernBERT cross-encoder (max proba):  {r['threshold']}")
  print(f"LLM:                     {r['llm_model']}")
  print(f"Поиск:                   {r['search_provider']}")
  print()
  print("--- Сравнение систем на всём val ---")
  print(f"RuModernBERT cross-encoder only:     acc={_fmt(r['bert_only_acc'])}  macro-F1={_fmt(r['bert_only_macro_f1'])}")
  print(f"Hybrid:        acc={_fmt(r['hybrid_acc'])}  macro-F1={_fmt(r['hybrid_macro_f1'])}")
  print(f"Δ hybrid−RuModernBERT cross-encoder: acc={delta_acc:+.4f}  macro-F1={delta_f1:+.4f}")
  print()
  print("--- Low-confidence (маршрут в агента) ---")
  print(f"Доля в агента:           {r['routed_to_agent_share']:.1%}")
  print(f"Agent acc (low-conf):    {_fmt(r['agent_acc_on_low_conf'])}")
  print(f"RuModernBERT cross-encoder baseline (low-conf): acc={_fmt(r['bert_low_conf_acc_baseline'])}  "
        f"macro-F1={_fmt(r['bert_low_conf_f1_baseline'])}")
  print()
  print("--- Поиск Tavily (среди вызовов агента) ---")
  print(f"Доля с поиском:          {r['search_used_share']:.1%}")
  print(f"Acc с поиском:           {_fmt(r.get('acc_with_search'))}")
  print(f"Acc без поиска:          {_fmt(r.get('acc_without_search'))}")
  print()
  print("--- Время и стоимость ---")
  print(f"Общее время:             {r['total_time_min']} мин")
  print(f"Latency mean:            {r['latency_mean_sec']} с")
  print(f"Latency median:          {r['latency_median_sec']} с")
  print(f"Latency с поиском:       {r['latency_with_search_sec']} с")
  print(f"Latency без поиска:      {r['latency_without_search_sec']} с")
  print(f"Токены prompt/completion:{r['total_prompt_tokens']:,} / {r['total_completion_tokens']:,}")
  print(f"Оценка стоимости:        {r['actual_cost_rub']} ₽")
  print()
  print("--- Ограничения ---")
  for i, lim in enumerate(r.get("known_limitations", []), 1):
    print(f"  {i}. {lim}")
  print("=" * 60)

print_stage4_report(report)

Stage 4 — общий отчёт (val)
Порог RuModernBERT cross-encoder (max proba):  0.68
LLM:                     deepseek/deepseek-v4-flash-alt (VseGPT)
Поиск:                   Tavily

--- Сравнение систем на всём val ---
RuModernBERT cross-encoder only:     acc=0.7690  macro-F1=0.7683
Hybrid:        acc=0.7738  macro-F1=0.7725
Δ hybrid−RuModernBERT cross-encoder: acc=+0.0048  macro-F1=+0.0042

--- Low-confidence (маршрут в агента) ---
Доля в агента:           20.3%
Agent acc (low-conf):    0.5849
RuModernBERT cross-encoder baseline (low-conf): acc=0.5708  macro-F1=0.5671

--- Поиск Tavily (среди вызовов агента) ---
Доля с поиском:          65.2%
Acc с поиском:           0.5738
Acc без поиска:          0.5186

--- Время и стоимость ---
Общее время:             137.85 мин
Latency mean:            8.636 с
Latency median:          5.939 с
Latency с поиском:       6.43 с
Latency без поиска:      4.555 с
Токены prompt/completion:2,073,102 / 28,321
Оценка стоимости:        85.19 ₽

--- Ограничения 

In [6]:
agent_df = pd.read_parquet(AGENT_LOW_CONF_PREDS_PATH)

print('Columns:', list(agent_df.columns))
print()
print('Routed to:')
print(agent_df[COL_ROUTED_TO].value_counts())
print()
print('Search used:', agent_df[COL_SEARCH_USED].sum(), '/', len(agent_df))
print('Parse errors (final_pred=-1):', (agent_df[COL_FINAL_PRED] == -1).sum())

agent_df.head()

Columns: ['final_pred', 'routed_to', 'search_used', 'search_query', 'prompt_tokens', 'completion_tokens', 'permalink', 'label', 'latency_sec']

Routed to:
routed_to
llm    925
Name: count, dtype: int64

Search used: 603 / 925
Parse errors (final_pred=-1): 48


,final_pred,routed_to,search_used,search_query,prompt_tokens,completion_tokens,permalink,label,latency_sec
0,1,llm,True,"Стекло24 Москва, Дмитровское шоссе, 13 защитно...",2556,26,85440605384,0,8.717
1,0,llm,True,Филиал № 3 ГУЗ Новомосковская городская клинич...,1761,61,62645894674,1,7.932
2,1,llm,True,Наоми Красноярск Свободный проспект 49 модное ...,3201,34,210122820967,1,6.541
3,1,llm,True,"ВодоходЪ Москва, Ленинградское шоссе, 51 речны...",2696,39,243733517910,0,11.865
4,0,llm,True,Магнит Краснодар Ставропольская улица 248 круг...,2570,23,1593256005,0,4.379
